In [ ]:
%pip install pandas 

# import json
# import time
# import uuid
# from datetime import datetime, timedelta

#this needs to be !pip install stuff

In [ ]:
import uuid
import json
from datetime import datetime, timedelta


class Application:
    def __init__(self, name):
        self.name = name


class Permission:
    def __init__(self, action, application):
        self.name = application.name + '.' + action
        self.action = action
        self.application = application

class Role:
    def __init__(self, name, parent=None):
        self.name = name
        self.permissions = []
        self.parent = parent

    def add_permission(self, permission):
        self.permissions.append(permission)

    def get_all_permissions(self):
        perms = list(self.permissions) #changed from set to list
        if self.parent:
            perms.extend(self.parent.get_all_permissions())
        return perms


class User:
    def __init__(self):
        self.id = str(uuid.uuid4())
        self.roles = []
        self.active_role = None
        self.session_start = None

    def assign_role(self, role):
        if len(self.roles) < 4:
            self.roles.append(role)
        else: 
            print(f"Error: {self.id} already has 4 roles.")

    def start_session(self, role_name):
        for role in self.roles: # User has to have role in order to impersonate
            if role.name == role_name: # If user has role, then...
                self.active_role = role
                self.session_start = datetime.now()
                print(f"{self.id} starts session with Role {role.name}")
                return

        print(f"ERROR: {self.id} cannot use role {role_name}.")


    def log_attempt(self, app_name, action, result):
        #log to json file
        log_entry = {
            "timestamp" : datetime.now().strftime("%a, %d, %b %Y %H: %M: %S GMT"), 
            "user" : self.id, 
            "application" : app_name, 
            "access_type" : action, 
            "result" : result
        }
        with open("access_log.json", "a") as file:
            file. write(json.dumps(log_entry) + "\n")

    def check_access(self, app_name, action):
        #session check
        if not self.active_role:
            print(f"Access Denied: {self.id} has no active session.")
            return
        
        #session expiry check
        if datetime.now() - self.session_start > timedelta(hours=1):
            print(f"ERROR: session has expired for {self.id}")
            return
        
        #permission check
        all_perms = self.active_role.get_all_permissions()
        for perm in all_perms:
            if p.application.name == app_name and p.action == action:
                print(f"Access granted: {self.id} to {app_name} ({action})")
                self.log_attempt(app_name, action, "Granted")
                return
        
        #access denied 
        print(f"Access denied: {self.id} for application = {app_name} ({action})")
        self.log_attempt(app_name, action, "Denied")


    #def is_session_valid(self):
    #    if not self.session_start:
    #        return False
    #    return datetime.now() - self.session_start < timedelta(hours=1) # Session expires after 1 hour

In [ ]:
# Normally these would go into a model as well but we don't have a typical service provider
def can(user, permission_name):

    # user_id, "MMI.Read"

    # 1. Active role check
    if not user.active_role:
        print(f"Access Denied: {user.id} -> No active session.")
        return False

    # 2. Session expiry check
    if not user.is_session_valid():
        print(f"ERROR: Session expired for {user.id}.")
        return False

    # 3. Split permission_name into application and action (maybe change this??
    try:
        app_name, action = permission_name.split('.', 1)
    except ValueError:
        print(f"ERROR: Invalid permission format '{permission_name}'. Use '[App].[Action]'.")
        return False

    # 4. Check permissions (includes inheritance)
    for perm in user.active_role.get_all_permissions():
        if perm.application.name == app_name and perm.action == action:
            print(f"Access Granted: {user.id} -> {perm.application.name} ({action})")
            return True

    # 5. Access denied
    print(f"Access Denied: {user.id} -> {app_name} ({action})")
    return False



# based on what the instructions seem to want for block 3

#the tables need to be in here as well
# Create applications
print("=== Seeding Applications... ===")
mmi = Application("Money Market Instruments")
derivatives = Application("Derivatives Trading")
interest = Application("Interest Instruments")
pci = Application("Private Consumer Instruments")
ibs = Application("Investment Banking Suite")
tm = Application("Treasury Management")
lm = Application("Loan Management")
rms = Application("Risk Management System")
ctp = Application("Corporate Trading Platform")
sms = Application("Share Management System")
ecg = Application("E-Commerce Gateway")
obap = Application("Office Banking Admin Panel")

# Create Permissions (all have RWX even if not used)
print("=== Seeding Permissions... ===")
mmi.Read = Permission("Read", mmi)
mmi.Write = Permission("Write", mmi)
MMI_Execute = Permission("Execute", mmi)

derivatives.Read = Permission("Read", derivatives)
derivatives.Write = Permission("Write", derivatives)
derivatives.Execute = Permission("Execute", derivatives)

interest.Read = Permission("Read", interest)
interest.Write = Permission("Write", interest)
interest.Execute = Permission("Execute", interest)

pci.Read = Permission("Read", pci)
pci.Write = Permission("Write", pci)
pci.Execute = Permission("Execute", pci)

ibs.Read = Permission("Read", ibs)
ibs.Write = Permission("Write", ibs)
ibs.Execute = Permission("Execute", ibs)
ibs.Approve = Permission("Approve", ibs) # Extra Perm

tm.Read = Permission("Read", tm)
tm.Write = Permission("Write", tm)
tm.Execute = Permission("Execute", tm)

lm.Read = Permission("Read", lm)
lm.Write = Permission("Write", lm)
lm.Execute = Permission("Execute", lm)

rms.Read = Permission("Read", rms)
rms.Write = Permission("Write", rms)
rms.Execute = Permission("Execute", rms)
rms.Approve = Permission("Approve", rms) # Extra Perm

ctp.Read = Permission("Read", ctp)
ctp.Write = Permission("Write", ctp)
ctp.Execute = Permission("Execute", ctp)
ctp.Audit = Permission("Audit", ctp) # Extra Perm
ctp.Report = Permission("Report", ctp) # Extra Perm

sms.Read = Permission("Read", sms)
sms.Write = Permission("Write", sms)
sms.Execute = Permission("Execute", sms)

ecg.Read = Permission("Read", ecg)
ecg.Write = Permission("Write", ecg)
ecg.Execute = Permission("Execute", ecg)

obap.Read = Permission("Read", obap)
obap.Write = Permission("Write", obap)
obap.Approve = Permission("Approve", obap) # Extra Perm
obap.Audit = Permission("Audit", obap) # Extra Perm


# Roles
print("=== Seeding Roles... ===")
role_a = Role('A')
role_b = Role('B', parent=role_a)
role_c = Role('C', parent=role_b)
role_d = Role('D', parent=role_c)
role_e = Role('E', parent=role_d)
role_f = Role('F', parent=role_e)
role_g = Role('G')
role_h = Role('H')
role_i = Role('I')

# Scenario 1: Successful Inheritance
u1 = User()
u1.assign_role(role_c)
u1.start_session("C")
u1.check_access('Money Market Instruments', 'Read') # Inherited from Role A

# Scenario 2: Access Denied (Unauthorized)
u1.check_access('Risk Management System', 'Approve')

# Scenario 3: Unauthorized Role Selection
u2 = User("Bob")
u2.assign_role(role_g)
u2.start_session("A") # Bob doesn't have Role A


### Seeding

In [ ]:
# Create applications
print("=== Seeding Applications... ===")
mmi = Application("Money Market Instruments")
derivatives = Application("Derivatives Trading")
interest = Application("Interest Instruments")
pci = Application("Private Consumer Instruments")
ibs = Application("Investment Banking Suite")
tm = Application("Treasury Management")
lm = Application("Loan Management")
rms = Application("Risk Management System")
ctp = Application("Corporate Trading Platform")
sms = Application("Share Management System")
ecg = Application("E-Commerce Gateway")
obap = Application("Office Banking Admin Panel")

# Create Permissions (all have RWX even if not used)
print("=== Seeding Permissions... ===")
mmi.Read = Permission("Read", mmi)
mmi.Write = Permission("Write", mmi)
MMI_Execute = Permission("Execute", mmi)

derivatives.Read = Permission("Read", derivatives)
derivatives.Write = Permission("Write", derivatives)
derivatives.Execute = Permission("Execute", derivatives)

interest.Read = Permission("Read", interest)
interest.Write = Permission("Write", interest)
interest.Execute = Permission("Execute", interest)

pci.Read = Permission("Read", pci)
pci.Write = Permission("Write", pci)
pci.Execute = Permission("Execute", pci)

ibs.Read = Permission("Read", ibs)
ibs.Write = Permission("Write", ibs)
ibs.Execute = Permission("Execute", ibs)
ibs.Approve = Permission("Approve", ibs) # Extra Perm

tm.Read = Permission("Read", tm)
tm.Write = Permission("Write", tm)
tm.Execute = Permission("Execute", tm)

lm.Read = Permission("Read", lm)
lm.Write = Permission("Write", lm)
lm.Execute = Permission("Execute", lm)

rms.Read = Permission("Read", rms)
rms.Write = Permission("Write", rms)
rms.Execute = Permission("Execute", rms)
rms.Approve = Permission("Approve", rms) # Extra Perm

ctp.Read = Permission("Read", ctp)
ctp.Write = Permission("Write", ctp)
ctp.Execute = Permission("Execute", ctp)
ctp.Audit = Permission("Audit", ctp) # Extra Perm
ctp.Report = Permission("Report", ctp) # Extra Perm

sms.Read = Permission("Read", sms)
sms.Write = Permission("Write", sms)
sms.Execute = Permission("Execute", sms)

ecg.Read = Permission("Read", ecg)
ecg.Write = Permission("Write", ecg)
ecg.Execute = Permission("Execute", ecg)

obap.Read = Permission("Read", obap)
obap.Write = Permission("Write", obap)
obap.Approve = Permission("Approve", obap) # Extra Perm
obap.Audit = Permission("Audit", obap) # Extra Perm


# Roles
print("=== Seeding Roles... ===")
role_a = Role('A')
role_b = Role('B', parent=role_a)
role_c = Role('C', parent=role_b)
role_d = Role('D', parent=role_c)
role_e = Role('E', parent=role_d)
role_f = Role('F', parent=role_e)
role_g = Role('G')
role_h = Role('H')
role_i = Role('I')

=== Seeding Applications... ===
=== Seeding Permissions... ===
=== Seeding Roles... ===
